# EDA — Voorrangswegen (OSM)

Explores whether OpenStreetMap contains enough information to classify Rotterdam intersections
as **voorrangsweg** (priority road) vs **niet-voorrangsweg**.

In the Netherlands, a voorrangsweg is marked by road sign B1 (haaientanden).
OSM does not have a dedicated `voorrangsweg` tag, but the `highway` classification
is a strong proxy:

| OSM `highway` value | Typically voorrangsweg? |
|---|---|
| `motorway`, `trunk`, `primary`, `secondary` | Yes |
| `tertiary` | Sometimes (worth checking) |
| `residential`, `living_street`, `unclassified` | No |

There is also a rarely-used `priority_road=yes/designated` tag — this notebook checks if it appears in Rotterdam.

**Goal:** Define a join strategy to attach a voorrangsweg label to each intersection in `intersections_merged.gpkg`,
so it can be added as a dimension in notebook 03.

In [ ]:
import os

PROJECT_DIR = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second year\Afstuderen\Project"

# Intersections file — used to test the join strategy at the end
INT_FILE = os.path.join(PROJECT_DIR, "data", "processed", "intersections_merged.gpkg")

# Cache: Rotterdam OSM edges saved locally to avoid re-downloading each run
OSM_CACHE = os.path.join(PROJECT_DIR, "data", "processed", "osm_edges_rotterdam.gpkg")

# Place name passed to OSMnx
PLACE = "Rotterdam, Netherlands"

# CRS for spatial operations — RD New (metres)
CRS_RD = "EPSG:28992"

print("Config loaded.")

## 1. Download Rotterdam road network from OSM

In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd

if os.path.exists(OSM_CACHE):
    # Load from cache to avoid hitting the OSM API on every run
    edges = gpd.read_file(OSM_CACHE)
    print(f"Loaded from cache: {len(edges):,} edges")
else:
    print("Downloading from OSM (first run only)...")
    # Request additional tags beyond the osmnx defaults — specifically priority_road
    # which is a rarely-used but explicit voorrangsweg tag in OSM
    ox.settings.useful_tags_way = ox.settings.useful_tags_way + ["priority_road", "bicycle_road"]
    G = ox.graph_from_place(PLACE, network_type="drive", retain_all=False)
    _, edges = ox.graph_to_gdfs(G)
    edges = edges.to_crs(CRS_RD)
    # Reset index so u/v/key become regular columns — easier to work with
    edges = edges.reset_index()
    edges.to_file(OSM_CACHE, driver="GPKG")
    print(f"Downloaded and cached: {len(edges):,} edges → {OSM_CACHE}")

print(f"\nColumns: {edges.columns.tolist()}")

## 2. Explore the `highway` attribute

`highway` is the primary OSM tag for road classification and the main proxy for voorrangsweg status.

In [ ]:
import matplotlib.pyplot as plt

# highway can be a list (when an edge has multiple tags) — explode to count individual values
hw_counts = edges["highway"].explode().value_counts()

print(f"Unique highway values: {len(hw_counts)}")
print(hw_counts.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
hw_counts.plot.bar(ax=ax, color="steelblue", edgecolor="white")
ax.set_title("OSM highway values — Rotterdam")
ax.set_ylabel("Edge count")
ax.set_xlabel("")
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 3. Check for explicit `priority_road` tag

Some mappers use `priority_road=yes` or `priority_road=designated` as an explicit tag.
This is rare but worth checking.

In [ ]:
if "priority_road" in edges.columns:
    print("priority_road tag found!")
    print(edges["priority_road"].value_counts(dropna=False))
else:
    print("No 'priority_road' column in OSM data for Rotterdam.")
    print("Will rely on 'highway' classification as proxy.")

## 4. Classify edges as voorrangsweg / niet-voorrangsweg

Based on the highway values found above, define a classification.
Adjust `VOORRANG_TYPES` if you want to include or exclude certain values.

In [ ]:
# Highway values that indicate a voorrangsweg (priority road) in a Dutch urban context.
# motorway/trunk are included for completeness but rarely appear at typical intersections.
# tertiary/tertiary_link added after visual inspection of the spot-check map.
VOORRANG_TYPES = {"motorway", "motorway_link", "trunk", "trunk_link",
                  "primary", "primary_link", "secondary", "secondary_link",
                  "tertiary", "tertiary_link"}

# For edges where highway is a list, check if any value is in VOORRANG_TYPES
def is_voorrangsweg(hw_val):
    if isinstance(hw_val, list):
        return any(v in VOORRANG_TYPES for v in hw_val)
    return hw_val in VOORRANG_TYPES

edges["voorrangsweg"] = edges["highway"].apply(is_voorrangsweg)

print("Voorrangsweg classification:")
print(edges["voorrangsweg"].value_counts())
print(f"
  {edges['voorrangsweg'].mean()*100:.1f}% of edges classified as voorrangsweg")

# Show which highway types ended up in each class
print("
Breakdown by highway type:")
print(
    edges.assign(hw=edges["highway"].apply(lambda x: x if not isinstance(x, list) else str(x)))
    .groupby(["hw", "voorrangsweg"])
    .size()
    .reset_index(name="n")
    .sort_values(["voorrangsweg", "n"], ascending=[False, False])
    .to_string(index=False)
)


## 5. Test join to intersections

Strategy: for each intersection, look at all road edges within a small buffer.
If any connected edge is a voorrangsweg, label the intersection as `voorrang`.

This mirrors how VRI and speed zone are assigned — by looking at connected segments.

In [ ]:
# Load intersections
intersections = gpd.read_file(INT_FILE)
print(f"Intersections: {len(intersections):,}")

# Buffer intersections to catch nearby edges
# 20m is enough to catch connected road segments without reaching the next block
BUFFER_M = 20
ints_buf = intersections[["JTE_ID", "geometry"]].copy()
ints_buf["geometry"] = ints_buf.buffer(BUFFER_M)

# Ensure both layers are in RD New
edges_rd = edges[["geometry", "voorrangsweg"]].copy()
if edges_rd.crs != CRS_RD:
    edges_rd = edges_rd.to_crs(CRS_RD)

# Spatial join: which intersection buffers intersect with a voorrangsweg edge?
joined = gpd.sjoin(
    edges_rd,
    ints_buf,
    how="inner",
    predicate="intersects"
)

# An intersection is voorrangsweg if ANY connected edge is a voorrangsweg
has_voorrang = set(
    joined[joined["voorrangsweg"] == True]["JTE_ID"].unique()
)

intersections["dim_voorrang"] = intersections["JTE_ID"].apply(
    lambda jte: "voorrang" if jte in has_voorrang else "geen_voorrang"
)

print("\ndim_voorrang distribution:")
print(intersections["dim_voorrang"].value_counts())
print(f"\n  {(intersections['dim_voorrang']=='voorrang').mean()*100:.1f}% of intersections on a voorrangsweg")

## 6. Spot-check: map of a small area

Visually verify that the classification looks correct for a known area.

In [ ]:
# Plot a 1.5km radius patch around Rotterdam Centrum to spot-check
# RD New coordinates for Rotterdam Centraal area
cx, cy = 92500, 436500  # approx centre of Rotterdam in RD New
RADIUS = 1500  # metres — increased from 1000 to see more roads

from shapely.geometry import box
bbox = box(cx - RADIUS, cy - RADIUS, cx + RADIUS, cy + RADIUS)

# Clip from the full edges (has both highway and voorrangsweg columns)
# edges_rd only kept geometry+voorrangsweg, so use edges which has everything
edges_clip = edges[edges.geometry.intersects(bbox)]
ints_clip  = intersections[intersections.geometry.within(bbox)]

# Helper: check if a highway value (or list) contains a specific type
def has_highway(hw_val, target):
    if isinstance(hw_val, list):
        return target in hw_val
    return hw_val == target

# Separate tertiary edges so we can colour them distinctly
is_tertiary = edges_clip["highway"].apply(lambda h: has_highway(h, "tertiary"))

fig, ax = plt.subplots(figsize=(10, 10))

# Roads: three layers — non-priority, tertiary (to evaluate), voorrangsweg
edges_clip[~edges_clip["voorrangsweg"] & ~is_tertiary].plot(
    ax=ax, color="lightgrey", linewidth=0.8, label="Overige wegen"
)
edges_clip[is_tertiary].plot(
    ax=ax, color="orange", linewidth=1.4, label="Tertiary (overweeg opnemen?)"
)
edges_clip[edges_clip["voorrangsweg"] == True].plot(
    ax=ax, color="steelblue", linewidth=2.0, label="Voorrangsweg (huidig)"
)

# Intersections: colour by dim_voorrang
ints_clip[ints_clip["dim_voorrang"] == "geen_voorrang"].plot(
    ax=ax, color="grey", markersize=4, zorder=3
)
ints_clip[ints_clip["dim_voorrang"] == "voorrang"].plot(
    ax=ax, color="red", markersize=6, zorder=4, label="Intersection op voorrangsweg"
)

ax.set_title("Voorrangswegen — Rotterdam Centrum (spot-check, r=1500m)")
ax.legend(loc="upper left", fontsize=8)
ax.set_axis_off()
plt.tight_layout()
plt.show()


## 7. Conclusions and next steps

**What we found:**
- No explicit `priority_road` tag in OSM Rotterdam data — `highway` is the only usable proxy.
- `VOORRANG_TYPES` = `{primary, secondary, trunk, motorway, *_link}` gives a reasonable classification.
- The 20m buffer join correctly identifies intersections on voorrangswegen.

**To add this dimension to notebook 03:**
1. Copy `VOORRAND_TYPES` and `BUFFER_M` to the notebook 03 config cell as `VOORRANG_TYPES` and `VOORRANG_BUFFER_M`.
2. Add a new section (Dimension 5) that:
   - Loads `osm_edges_rotterdam.gpkg` (the cache created in this notebook)
   - Runs the same buffer → sjoin → `dim_voorrang` logic above
3. Add `USE_VOORRANG = True` toggle alongside the other `USE_*` flags.
4. Add `"dim_voorrang"` to the list of possible dimensions in notebook 05's `ALL_DIM_COLS`.

**Potential issue to watch:** OSM coverage of `secondary` roads in Rotterdam is good,
but some smaller voorrangswegen may only be tagged `tertiary` — check the spot-check map
to decide whether to include `tertiary` in `VOORRANG_TYPES`.